In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import warnings 
warnings.filterwarnings("ignore")

In [ ]:
from DFTStructureGenerator import B_N_Cl, xtb_process, mol_manipulation, logfile_process, Tool
import glob, os, shutil
from rdkit import Chem
import numpy as np
from tqdm import tqdm
import pandas as pd

In [6]:
# DFT Method In Need
#p opt=(maxcycles=150) freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm
OPT_METHOD = "opt freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"
SPE_METHOD = "wb97xd/6-311+g(d,p) scrf=(smd,solvent=toluene) nosymm"
SPE_WFN_METHOD = "wb97xd/6-311+g(d,p) scrf=(smd,solvent=toluene) nosymm output=wfn"
OM_METHOD = "opt=(modredundant,maxcycles=64) freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"
TS_METHOD = "opt=(calcfc,ts,noeigen,maxcycles=64) freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"
SPE_SPECIAL = "b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"

In [ ]:
# root_file = "G:/work/B_Cl_Nu/Data" # Core directory for data storage
reactant_dir = "G:/work/B_Cl_Nu/Data" 
mol_dir = os.path.join(reactant_dir, 'Mols')              # Directory where Mol files are stored
dft_dir = os.path.join(reactant_dir, 'GS_OPT')             # Directory to store DFT data
spe_dir = os.path.join(reactant_dir, 'GS_SPE')                # Directory to store SPE data

# Data Supplement At least BNCl_Index and Cl_atomid are required

In [9]:
BN_df = pd.read_csv('Data/All_Data/reactants_B_N_passed.csv')
Cl_df = pd.read_csv('Data/All_Data/reactants_Cl_passed.csv')
def complete_target(data_csv, name='other_multi'):
    first_ts = {"B_smiles":{}, "B_Index":{}, "B_Atomid":{}, "N_smiles":{}, "N_Index":{}, "N_Atomid":{}, "Cl_smiles":{}, "Cl_Index":{}, 
    "Cl_Atomid":{}, "B_N_Cl_conf":{}, "Cl_r_conf":{}, "deltaG_react":{}, "G_energy":{}}
    first_ts_idx = 0
    for row_id, row in data_csv.iterrows():
        B_Index, N_Index, Cl_Index = row['B_Index'], row['N_Index'], row['Cl_Index']
        # B_Index, N_Index, Cl_Index, Cl_Atomid = row['B_Index'], row['N_Index'], row['Cl_Index'], row['Cl_Atomid']
        first_ts["B_Index"][first_ts_idx] = int(B_Index)
        first_ts["N_Index"][first_ts_idx] = int(N_Index)
        first_ts["Cl_Index"][first_ts_idx] = int(Cl_Index)
        # first_ts["Cl_Atomid"][first_ts_idx] = int(Cl_Atomid)
        BN_id = BN_df[(BN_df['B_Index'] == B_Index) & (BN_df['N_Index'] == N_Index)].index[0]
        Cl_id = Cl_df[Cl_df['Index'] == Cl_Index].index[0]
        first_ts["Cl_Atomid"][first_ts_idx] = Cl_df['Atomid'][Cl_id]
        # Cl_id = Cl_df[(Cl_df['Index'] == Cl_Index) & (Cl_df['Atomid'] == Cl_Atomid)].index[0]
        first_ts["B_smiles"][first_ts_idx] = BN_df['B_smiles'][BN_id]
        first_ts["B_Atomid"][first_ts_idx] = int(BN_df['B_Atomid'][BN_id])
        first_ts["N_smiles"][first_ts_idx] = BN_df['N_smiles'][BN_id]
        first_ts["N_Atomid"][first_ts_idx] = int(BN_df['N_Atomid'][BN_id])
        first_ts["Cl_smiles"][first_ts_idx] = Cl_df['Smiles'][Cl_id]
        first_ts["B_N_Cl_conf"][first_ts_idx] = int(BN_df['conf_idxs_p'][BN_id])
        first_ts["Cl_r_conf"][first_ts_idx] = int(Cl_df['conf_idxs_r'][Cl_id])
        first_ts["deltaG_react"][first_ts_idx] = 627.5 * (BN_df['deltaG_react'][BN_id] + Cl_df['deltaG_react'][Cl_id])
        first_ts["G_energy"][first_ts_idx] = BN_df['G_energy_r'][BN_id] + Cl_df['G_energy_r'][Cl_id]
        first_ts_idx += 1
    first_df = pd.DataFrame(first_ts)
    first_df.to_csv(f'Data/Iteration_Data/{name}.csv', index=False)

In [ ]:
complete_target(pd.read_csv(f'E:/work/B_Cl_Nu/Sum/result_clear.csv'), name='result_clear')

In [ ]:
name = "Sum"

In [ ]:
target_dir = os.path.join(r'E:\work\B_Cl_Nu', name)
mol_xtb_file = os.path.join(target_dir, 'OM_XTB')
new_mol_dir = os.path.join(target_dir, 'Mols')
Cl_r_file = os.path.join(dft_dir, 'Cl_r')
B_N_p_file = os.path.join(dft_dir, 'B_N_p')
B_N_p_d_file = os.path.join(dft_dir, 'B_N_p_d')
mol_dir = os.path.join(root_file, 'Mols')
mol_xtb_file = os.path.join(target_dir, 'OM_XTB')
dft_dir_ = target_dir + "/OM"
ts_dir_ = target_dir + "/TS"
test_dir = target_dir + "/Test"


In [26]:
duplicate_N_id = [9, 43, 285, 310, 314, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 372, 375, 376]
duplicate_Cl_id = [624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636,
       637, 638, 639, 640, 642, 644, 645, 652, 653, 654, 655, 656, 657, 658, 659, 660, 661, 662, 663, 664,
       665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 675, 676, 677,
       678, 679, 680, 681, 682, 683, 684, 685, 686, 687, 688, 689, 690,
       691, 692, 693, 694, 695, 696, 697, 698, 699, 700, 701, 702, 703,
       704, 705, 706, 707, 708, 709, 710, 711, 713, 714, 716, 717, 718, 719, 720, 721, 722]

In [ ]:
# XTB optimization
all_results_ = []
if not os.path.isdir(target_dir):
    os.mkdir(target_dir)
if not os.path.isdir(mol_xtb_file):
    os.mkdir(mol_xtb_file)
if not os.path.isdir(new_mol_dir):
    os.mkdir(new_mol_dir)
reaction_csv_path = f'Data/Dataset/{name}.csv'
reaction_csv = pd.read_csv(reaction_csv_path)
all_react_names, all_react_mols, all_restrict = [], [], []
for ix, row in reaction_csv.iterrows():
    B_Index, N_Index, Cl_Index = row['B_Index'], row['N_Index'], row['Cl_Index']
    sum_nwmol, react_name, restrict, group_a, group_b = B_N_Cl.generate_ts_structure(row, 2.6, 2.0, target_dir=target_dir, reactant_dir=reactant_dir, duplicate_N_id=duplicate_N_id, duplicate_Cl_id=duplicate_Cl_id)
    if sum_nwmol is None:
        continue
    Chem.MolToMolFile(sum_nwmol, os.path.join(new_mol_dir, f"{react_name}.mol"), kekulize=False, )
    all_react_names.append(react_name)
    all_react_mols.append(sum_nwmol)
    all_restrict.append(restrict)
xtb_process.xtb_main(all_react_names, all_react_mols, dir_path=mol_xtb_file, core=50, restrict=all_restrict)

In [17]:
xtb_process.shift_to_parra(mol_xtb_file, 0, 1)

In [ ]:
# restrictive optimization
if not os.path.isdir(dft_dir_):
    os.mkdir(dft_dir_)
B_N_Cl.smiles_DFT_calc(mol_xtb_file, new_mol_dir, dft_dir_, method = OM_METHOD, conf_limit = 3, SpinMultiplicity=2) 
for name_, mol, restrict in zip(all_react_names, all_react_mols, all_restrict):
    gjfs = glob.glob(os.path.join(dft_dir_, f"{name_}*.gjf"))
    for gjf in gjfs:
        atoms, positions = B_N_Cl.FormatConverter.read_gjf(gjf)
        title = " ".join([str(each) for each in [restrict[0][0], restrict[0][1], restrict[1][1]]])
        freeze = restrict[:2]
        B_N_Cl.FormatConverter.block_to_gjf(atoms, positions, gjf, 0, 2, title, OM_METHOD, freeze = freeze, )

In [ ]:
# transition state optimization
mol_manipulation.B_N_Cl.reaction_calc_ts(target_dir, method=TS_METHOD, om_name='OM', ts_name='TS')

In [ ]:
# transition state secondary optimization
bond_attach_function = B_N_Cl.logfile_process.bond_addition_function()
bond_attach_function.add_function([0,1], 3.5, 'less')
bond_attach_function.add_function([1,2], 2.5, 'less')
bond_ignore_list = [[0, 1, 2]]
mol_manipulation.error_improve(target_dir, new_mol_dir, 'TS', 'ts_dust_bin', improve_dir='TS_improve', bond_attach_std='mol', bond_addition_function=bond_attach_function, bond_ignore_list = bond_ignore_list, maxcycles=100)

In [ ]:
bond_attach_function = B_N_Cl.logfile_process.bond_addition_function()
bond_attach_function.add_function([0,1], 3.5, 'less')
bond_attach_function.add_function([1,2], 2.5, 'less')
bond_ignore_list = [[0, 1, 2]]
mol_manipulation.error_improve(target_dir, new_mol_dir, 'TS_Finish2', 'ts_dust_bin', improve_dir='TS_improve', bond_attach_std='mol', bond_addition_function=bond_attach_function, bond_ignore_list = bond_ignore_list, maxcycles=100)

In [ ]:
B_N_Cl.SPE_DFT_calc(target_dir, 'TS', 'TS_ENG', method=SPE_METHOD)

In [ ]:
bond_attach_function = B_N_Cl.logfile_process.bond_addition_function()
bond_ignore_list = [[0, 1, 2]]
mol_manipulation.error_improve(target_dir, new_mol_dir, 'TS_ENG', 'TS_ENG_dustbin', improve_dir='TS_ENG_imp', bond_attach_std='mol', bond_addition_function=bond_attach_function, bond_ignore_list = bond_ignore_list, maxcycles=100)


In [23]:
bond_attach_function = B_N_Cl.logfile_process.bond_addition_function()
bond_ignore_list = [[0, 1, 2]]
mol_manipulation.error_improve(target_dir, new_mol_dir, 'TS_ENG', 'TS_ENG_dustbin', improve_dir='TS_ENG_imp', bond_attach_std='mol', bond_addition_function=bond_attach_function, bond_ignore_list = bond_ignore_list, maxcycles=100)
B_N_Cl.collection_dft_ts(ts_csv_path=f'Data/Iteration_Data/{name}.csv', dft_dir=target_dir + "/TS", spe_dir=target_dir + "/TS_ENG", duplicate_Cl_id=duplicate_Cl_id)

31it [00:01, 23.47it/s]


# IRC Analysis

In [ ]:
all_B_id = BN_df['B_Index'].unique().tolist()
all_N_id = BN_df['N_Index'].unique().tolist()
all_Cl_id = Cl_df['Index'].unique().tolist()

In [ ]:
avaliable_TS_files = []
for each_file in glob.glob(r"E:\work\B_Cl_Nu\Sum\TS_ENG\*.log"):
    B_id, N_id, Cl_id = [int(each) for each in os.path.basename(each_file).split('.')[0].split('_')[1:6:2]]
    if B_id in all_B_id and N_id in all_N_id and Cl_id in all_Cl_id:
        # avaliable_TS_files.append(each_file)
        avaliable_TS_files.append([B_id, N_id, Cl_id])
avaliable_TS_files = np.unique(np.array(avaliable_TS_files), axis=0)
print(len(avaliable_TS_files))

In [ ]:
target_dir = r"E:\work\B_Cl_Nu\Sum"
need_irc_files = B_N_Cl.prepare_irc_jobs(
    avaliable_TS_files=avaliable_TS_files,
    target_dir=target_dir,
    ts_glob_template=r"E:/work/B_Cl_Nu/Sum/TS/B_{B_id:05}_Nu_{N_id:05}_Cl_{Cl_id:05}*.log",
    ts_name="TS",
    ts_need_irc_name="TS_needIRC",
    ts_energy_name="TS_ENG",
    require_all=True,
)


In [ ]:
# Ensure that both IRC endpoints form at least one target bond and cover both directions
moved_fail_logs = B_N_Cl.move_failed_irc_logs(
    target_dir=r"E:\work\B_Cl_Nu\Sum",
    ts_name="TS_needIRC",
    irc_name="IRC_full",
    fail_dir_name="TS_fail_IRC",
    distance_1_max=2.2,
    distance_2_max=2.1,
)


In [ ]:
# Ensure that one IRC endpoint corresponds to B-Cl bonding and the other to C-Cl bonding
moved_uncertain_logs = B_N_Cl.move_uncertain_irc_logs(
    target_dir=r"E:\work\B_Cl_Nu\Sum",
    ts_name="TS_needIRC",
    irc_name="IRC_full",
    uncertain_dir_name="TS_uncertain_IRC",
    distance_1_bond_max=2.2,
    distance_2_broken_min=2.5,
    distance_1_broken_min=2.6,
    distance_2_bond_max=2.1,
)


# Summary

In [ ]:
BN_energies_r = {f'{row['B_Index']:05}_{row['N_Index']:05}':row['G_energy_r'] for _, row in BN_df.iterrows()}
Cl_energies_r = {f'{row['Index']:05}':row['G_energy_r'] for _, row in Cl_df.iterrows()}
BN_energies_p = {f'{row['B_Index']:05}_{row['N_Index']:05}':row['G_energy_p'] for _, row in BN_df.iterrows()}
Cl_energies_p = {f'{row['Index']:05}':row['G_energy_p'] for _, row in Cl_df.iterrows()}

In [ ]:
result_df = B_N_Cl.export_ts_summary_csv(
    target_dir=r"E:\work\B_Cl_Nu\Sum",
    bn_energies_r=BN_energies_r,
    cl_energies_r=Cl_energies_r,
    bn_energies_p=BN_energies_p,
    cl_energies_p=Cl_energies_p,
    output_csv=r"E:/work/B_Cl_Nu/Sum/result.csv",
    ts_name="TS_needIRC",
    ts_eng_name="TS_ENG",
)


In [ ]:
target_csv = B_N_Cl.annotate_reaction_aam_csv(
    target_csv_path=r"E:/work/B_Cl_Nu/Sum/result_clear.csv",
    output_csv_path=r"E:/work/B_Cl_Nu/Sum/result_clear_AAM.csv",
)
